In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Cell 1 — Setup

In [ ]:
import os
import glob
import json
import pandas as pd
from collections import Counter
from google.colab import drive

drive.mount('/content/drive')

BASE_PATH = "/content/drive/MyDrive/Transcripts_CSS"
OUTPUT_DIR = os.path.join(BASE_PATH, "outputs")
STANZA_DIR = os.path.join(OUTPUT_DIR, "stanza_annotations")
LDA_DIR = os.path.join(OUTPUT_DIR, "LDA_results")

COMMUNITY_SHORT_NAME = {
    "transcripts_business": "business",
    "transcripts_religion": "religion",
    "transcripts_comedy": "comedy",
    "transcripts_lifestyle": "lifestyle",
    "transcripts_tech": "tech",
    "politics_transcripts": "politics",
    "gaming_transcripts": "gaming",
    "motivational_transcripts": "motivational",
}

pd.set_option('display.max_columns', None)
os.makedirs(OUTPUT_DIR, exist_ok=True)

Cell 2 — Extract per-video UPOS + deprel raw counts (the df-parse.csv equivalent)

In [ ]:
def extract_counts(stanza_annotations):
    """
    Counts raw occurrences of each UPOS tag and each deprel value
    across a video's token list — the Stanza equivalent of the base
    paper's parse_NP_count etc.
    """
    upos_counter = Counter()
    deprel_counter = Counter()

    for token in stanza_annotations:
        upos = token.get("upos")
        deprel = token.get("deprel")
        if upos:
            upos_counter[upos] += 1
        if deprel:
            deprel_counter[deprel] += 1

    return upos_counter, deprel_counter

records = []
all_upos_tags = set()
all_deprel_tags = set()

for community, short_name in COMMUNITY_SHORT_NAME.items():
    pattern = os.path.join(STANZA_DIR, f"stanza_{short_name}", f"stanza_{short_name}_full_annotations.json")
    matches = glob.glob(pattern)

    if not matches:
        print(f"WARNING: no full_annotations.json found for {community}")
        continue

    with open(matches[0], encoding="utf-8") as f:
        data = json.load(f)

    for entry in data:
        upos_counter, deprel_counter = extract_counts(entry.get("stanza_annotations", []))
        all_upos_tags.update(upos_counter.keys())
        all_deprel_tags.update(deprel_counter.keys())

        record = {
            "video_id": entry["video_id"],
            "hindi_pct": entry.get("hi_script_ratio", 0.0) * 100,   # normalize to 0-100 like df.csv
            "english_pct": entry.get("en_script_ratio", 0.0) * 100,
            "n_tokens_original": entry.get("n_tokens_original", 0),
            "n_tokens_cleaned": entry.get("n_tokens_cleaned", 0),
            "n_tokens_annotated": entry.get("n_tokens_annotated", 0),
            "n_discourse_markers": entry.get("n_discourse_markers", 0),
            "upos_counter": upos_counter,
            "deprel_counter": deprel_counter,
        }
        records.append(record)

print(f"Total videos processed: {len(records)}")
print(f"Distinct UPOS tags found: {sorted(all_upos_tags)}")
print(f"Distinct deprel tags found: {sorted(all_deprel_tags)}")

Cell 3 — Flatten counters into columns (upos_NOUN_count, deprel_root_count, etc.)

In [ ]:
rows = []
for r in records:
    row = {
        "video_id": r["video_id"],
        "hindi_pct": r["hindi_pct"],
        "english_pct": r["english_pct"],
        "n_tokens_original": r["n_tokens_original"],
        "n_tokens_cleaned": r["n_tokens_cleaned"],
        "n_tokens_annotated": r["n_tokens_annotated"],
        "n_discourse_markers": r["n_discourse_markers"],
    }
    for tag in all_upos_tags:
        row[f"upos_{tag}_count"] = r["upos_counter"].get(tag, 0)
    for tag in all_deprel_tags:
        row[f"deprel_{tag}_count"] = r["deprel_counter"].get(tag, 0)
    rows.append(row)

stanza_features_df = pd.DataFrame(rows)
print(f"Stanza features df: {stanza_features_df.shape}")
print(f"Duplicate video_ids: {stanza_features_df['video_id'].duplicated().sum()}")

output_path = os.path.join(OUTPUT_DIR, "df-stanza-features.csv")
stanza_features_df.to_csv(output_path, index=False)
print(f"Saved: {output_path}")
stanza_features_df.head(3)